## 1. Imports & Configuration

In [ ]:
import os
import json
import random
import re
import torch
from pathlib import Path
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from convokit import Corpus, download

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────

PRED_MODEL_ID  = "Qwen/Qwen3.5-9B-Base"        # base model for prediction (no chat template)
JUDGE_MODEL_ID = "Qwen/Qwen2.5-14B-Instruct" # instruct model for judging

HF_TOKEN=""

N_CONVERSATIONS = 50
RANDOM_SEED     = 42   # same seed as simulation notebook → same 50 conversations

# Generation hyper-parameters for the base (prediction) model
PRED_MAX_NEW_TOKENS = 200
PRED_TEMPERATURE    = 0.7
PRED_TOP_P          = 0.9

# Generation hyper-parameters for the judge
JUDGE_MAX_NEW_TOKENS = 1024
JUDGE_TEMPERATURE    = 0.3

# Output directories
BASE_DIR   = Path("../data/persuasion_prediction")
PRED_DIR   = BASE_DIR / "predictions"
JUDGE_DIR  = BASE_DIR / "judge_annotations"
PRED_DIR.mkdir(parents=True, exist_ok=True)
JUDGE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Prediction output : {PRED_DIR}")
print(f"Judge output      : {JUDGE_DIR}")

## 2. Load Prediction Model (Base — No Chat Template)

In [ ]:
print(f"Loading prediction model: {PRED_MODEL_ID}")

pred_tokenizer = AutoTokenizer.from_pretrained(PRED_MODEL_ID)
pred_model     = AutoModelForCausalLM.from_pretrained(
    PRED_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN
)
pred_model.eval()

if pred_tokenizer.pad_token is None:
    pred_tokenizer.pad_token = pred_tokenizer.eos_token

print("Prediction model loaded.")

## 3. Load Dataset

In [ ]:
print("Downloading / loading Persuasion for Good corpus...")
corpus = Corpus(filename=download("persuasionforgood-corpus"))

all_convos = list(corpus.iter_conversations())
print(f"Total conversations in corpus: {len(all_convos)}")

## 4. Helper Functions — Utterance Ordering & Role Detection

In [ ]:
def get_speaker_role(utterance):
    """Return 'Persuader' (role=0) or 'Persuadee' (role=1) from utterance meta."""
    role = utterance.meta.get("role")
    if role == 0:
        return "Persuader"
    elif role == 1:
        return "Persuadee"
    return ""


def _utt_sort_key(utt):
    """Sort by (user_turn_id, role) so within each turn persuader (0) comes first."""
    turn_id = utt.meta.get("user_turn_id", 0)
    role    = utt.meta.get("role", 0)
    return (turn_id, role)


def get_utterances_in_order(conversation):
    """Return utterances sorted by (user_turn_id, role)."""
    utts = list(conversation.iter_utterances())
    return sorted(utts, key=_utt_sort_key)


def get_ordered_turns(conversation):
    """
    Group utterances into (persuader_message, human_response) pairs.
    Consecutive same-role utterances are concatenated into one block.
    """
    utts = get_utterances_in_order(conversation)
    turns = []
    i = 0
    while i < len(utts):
        role = get_speaker_role(utts[i])
        if role == "Persuader":
            persuader_parts = []
            while i < len(utts) and get_speaker_role(utts[i]) == "Persuader":
                persuader_parts.append(utts[i].text.strip())
                i += 1
            persuader_msg = " ".join(persuader_parts)

            persuadee_parts = []
            while i < len(utts) and get_speaker_role(utts[i]) == "Persuadee":
                persuadee_parts.append(utts[i].text.strip())
                i += 1
            human_response = " ".join(persuadee_parts)

            if persuader_msg and human_response:
                turns.append({
                    "persuader_message": persuader_msg,
                    "human_response": human_response,
                })
        else:
            i += 1
    return turns


def get_persuadee(conversation):
    """Return the Speaker object for the persuadee in the conversation."""
    for utt in get_utterances_in_order(conversation):
        if get_speaker_role(utt) == "Persuadee":
            return utt.speaker
    return None


def has_persona_data(speaker):
    """Check whether the speaker has at least one demographic or personality field."""
    if speaker is None:
        return False
    meta = speaker.meta
    return any(meta.get(k) not in (None, "", "N/A") for k in ["age", "sex", "race", "edu", "extrovert"])

## 5. Select 50 Conversations

In [ ]:
random.seed(RANDOM_SEED)

valid_convos = []
for conv in all_convos:
    turns     = get_ordered_turns(conv)
    persuadee = get_persuadee(conv)
    if len(turns) >= 2 and has_persona_data(persuadee):
        valid_convos.append(conv)

print(f"Valid conversations (≥2 turns + persona data): {len(valid_convos)}")

selected_convos = random.sample(valid_convos, min(N_CONVERSATIONS, len(valid_convos)))
print(f"Selected {len(selected_convos)} conversations for prediction.")

## 6. Persona Extraction

In [ ]:
# Exact metadata keys confirmed from corpus inspection
DEMO_KEYS = {
    "age": "age", "sex": "sex", "race": "race", "edu": "education",
    "marital": "marital_status", "employment": "employment",
    "income": "income", "religion": "religion", "ideology": "ideology",
}
BIG5_KEYS = {
    "extrovert": "extroversion", "agreeable": "agreeableness",
    "conscientious": "conscientiousness", "neurotic": "neuroticism", "open": "openness",
}
MF_KEYS = {
    "care": "care", "fairness": "fairness", "loyalty": "loyalty",
    "authority": "authority", "purity": "purity", "freedom": "freedom",
}
SCHWARTZ_KEYS = {
    "conform": "conformity", "tradition": "tradition", "benevolence": "benevolence",
    "universalism": "universalism", "self_direction": "self_direction",
    "stimulation": "stimulation", "hedonism": "hedonism", "achievement": "achievement",
    "power": "power", "security": "security",
}
DM_KEYS = {
    "rational": "rational", "intuitive": "intuitive",
}


def _extract_group(meta, key_map):
    """Extract available fields using {raw_key: label} map; skip missing/null values."""
    return {
        label: meta[raw_key]
        for raw_key, label in key_map.items()
        if meta.get(raw_key) not in (None, "", "N/A")
    }


def extract_persona(speaker):
    """Return a fully structured persona dict from a ConvoKit Speaker."""
    meta = speaker.meta
    return {
        "speaker_id"       : speaker.id,
        "demographics"     : _extract_group(meta, DEMO_KEYS),
        "big_five"         : _extract_group(meta, BIG5_KEYS),
        "moral_foundations": _extract_group(meta, MF_KEYS),
        "schwartz_values"  : _extract_group(meta, SCHWARTZ_KEYS),
        "decision_making"  : _extract_group(meta, DM_KEYS),
    }

## 7. Prompt Building — Prediction Framing (No Chat Template)

In [ ]:
def persona_to_text(persona):
    """Convert structured persona dict to a readable natural-language description."""
    lines = []

    d = persona.get("demographics", {})
    if d:
        lines.append("Demographics — " + ", ".join(f"{k}: {v}" for k, v in d.items()) + ".")

    b5 = persona.get("big_five", {})
    if b5:
        lines.append("Big Five personality (scale 1–5) — " + ", ".join(f"{k} = {v}" for k, v in b5.items()) + ".")

    mf = persona.get("moral_foundations", {})
    if mf:
        lines.append("Moral foundations (scale 1–6) — " + ", ".join(f"{k} = {v}" for k, v in mf.items()) + ".")

    sv = persona.get("schwartz_values", {})
    if sv:
        lines.append("Schwartz values (scale 1–6) — " + ", ".join(f"{k} = {v}" for k, v in sv.items()) + ".")

    dm = persona.get("decision_making", {})
    if dm:
        lines.append("Decision-making style (scale 1–5) — " + ", ".join(f"{k} = {v}" for k, v in dm.items()) + ".")

    return "\n".join(lines) if lines else "No persona data available."


def build_prediction_prompt(persona, history, persuader_message):
    """
    Build a plain-text completion prompt for the base model.
    The model is asked to PREDICT the persuadee's response as an outside analyst,
    not to act as the persuadee.
    History uses real human responses to keep context grounded.
    """
    persona_text = persona_to_text(persona)

    prompt = (
        "The following is a conversation between a Persuader and a Persuadee.\n"
        "The Persuader is trying to convince the Persuadee to donate to a charity called 'Save the Children'.\n\n"
        "Persuadee's background:\n"
        f"{persona_text}\n\n"
        "Task: Based on the Persuadee's background and the conversation history below, "
        "predict what the Persuadee will say next in response to the Persuader's message. "
        "Output only the predicted response (1–4 sentences). "
        "Do not explain your reasoning.\n\n"
        "Conversation history:\n"
    )

    for turn in history:
        prompt += f"Persuader : {turn['persuader']}\n"
        prompt += f"Persuadee : {turn['persuadee']}\n"

    prompt += f"Persuader : {persuader_message}\n"
    prompt += "Persuadee :"

    return prompt

## 8. Generation Function (Standard Completion)

In [ ]:
def parse_model_output(raw_text, stop_strings=()):
    """
    Extract <think>...</think> reasoning and the response that follows it.
    If no <think> block is present, reasoning is empty and the full text is the response.
    Stop strings are applied to the response portion only.

    Returns: {"reasoning": str, "response": str}
    """
    reasoning = ""
    response  = raw_text

    think_match = re.search(r"<think>(.*?)</think>", raw_text, re.DOTALL)
    if think_match:
        reasoning = think_match.group(1).strip()
        response  = raw_text[think_match.end():].strip()

    for stop in stop_strings:
        if stop in response:
            response = response[:response.index(stop)].strip()

    return {"reasoning": reasoning, "response": response}


def generate_completion(prompt, model, tokenizer,
                        max_new_tokens=PRED_MAX_NEW_TOKENS,
                        temperature=PRED_TEMPERATURE,
                        top_p=PRED_TOP_P,
                        stop_strings=("\nPersuader :", "\nPersuadee :")):
    """
    Run standard text completion on `prompt` using the base model.
    No chat template is applied.
    Returns a dict: {"reasoning": <think> block content, "response": final response text}
    """
    inputs    = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs    = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    raw_text = tokenizer.decode(
        output_ids[0][input_len:],
        skip_special_tokens=True
    ).strip()

    return parse_model_output(raw_text, stop_strings=stop_strings)

## 9. Predict a Single Conversation

In [ ]:
def predict_conversation(conversation, model, tokenizer):
    """
    Run turn-by-turn prediction for one conversation.
    Returns a result dict ready to be saved as JSON.

    History is built using the REAL human responses so that the persuader's
    subsequent messages remain contextually grounded.
    """
    persuadee = get_persuadee(conversation)
    persona   = extract_persona(persuadee)
    turns     = get_ordered_turns(conversation)

    history      = []   # list of {persuader, persuadee} — populated with real human replies
    result_turns = []

    for turn_idx, turn in enumerate(turns):
        persuader_msg  = turn["persuader_message"]
        human_response = turn["human_response"]

        prompt        = build_prediction_prompt(persona, history, persuader_msg)
        llm_prediction = generate_completion(prompt, model, tokenizer)

        result_turns.append({
            "turn_index"       : turn_idx,
            "persuader_message": persuader_msg,
            "human_response"   : human_response,
            "llm_prediction"   : llm_prediction,
        })

        # Advance history using the REAL human response
        history.append({"persuader": persuader_msg, "persuadee": human_response})

    return {
        "conversation_id"  : conversation.id,
        "persuadee_id"     : persuadee.id,
        "persuadee_persona": persona,
        "turns"            : result_turns,
    }

## 10. Run Prediction on 50 Conversations & Save JSONs

In [ ]:
def save_prediction(result, output_dir):
    conv_id  = result["conversation_id"]
    out_path = output_dir / f"conv_{conv_id}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    return out_path


failed = []

for conv in tqdm(selected_convos, desc="Predicting conversations"):
    try:
        result   = predict_conversation(conv, pred_model, pred_tokenizer)
        out_path = save_prediction(result, PRED_DIR)
        print(f"  Saved: {out_path.name}  ({len(result['turns'])} turns)")
    except Exception as e:
        print(f"  ERROR on {conv.id}: {e}")
        failed.append(conv.id)

print(f"\nDone. {len(selected_convos) - len(failed)}/{len(selected_convos)} conversations saved.")
if failed:
    print(f"Failed: {failed}")

---
## 11. Judge — Load Qwen2.5-14B-Instruct

In [ ]:
# Free prediction model memory before loading the judge (optional but recommended
# if running on a single GPU)
# del pred_model
# torch.cuda.empty_cache()

print(f"Loading judge model: {JUDGE_MODEL_ID}")

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
judge_model     = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
judge_model.eval()

print("Judge model loaded.")

## 12. Judge Prompt & Generation

In [ ]:
JUDGE_SYSTEM_PROMPT = """\
You are an expert conversation analyst specialising in persuasion dynamics and human behaviour prediction.
You will be given a conversation between a Persuader and a Persuadee, displayed turn by turn.
For each turn you are shown the ACTUAL human response and an LLM-predicted response.

Your task is to produce a structured JSON analysis with the following schema:

{
  "conversation_id": "<id>",
  "turn_annotations": [
    {
      "turn_index": <int>,
      "similarity": "<describe ways the predicted response matches the actual — tone, stance, content, phrasing>",
      "distinctions": "<describe key differences — what did the LLM miss, overstate, or get wrong>",
      "prediction_quality": "<one of: excellent | good | partial | poor>"
    }
  ],
  "overall_assessment": {
    "best_predicted_turns": [<list of turn_index with highest accuracy>],
    "worst_predicted_turns": [<list of turn_index with lowest accuracy>],
    "summary": "<2–3 sentence overall assessment of how well the LLM predicted this person's responses>"
  }
}

Guidelines:
- Focus on semantic and pragmatic similarity, not just lexical overlap.
- Note if the prediction captures the right emotional register (e.g. sceptical, warm, defensive).
- Note if the prediction correctly captures stance toward donating (resistant, neutral, open).
- Return ONLY the JSON object. No extra text before or after."""


def build_judge_user_message(prediction_result):
    """Format the prediction result as a readable conversation block for the judge."""
    conv_id = prediction_result["conversation_id"]
    lines   = [f"Conversation ID: {conv_id}\n"]

    for t in prediction_result["turns"]:
        lines.append(f"--- Turn {t['turn_index']} ---")
        lines.append(f"Persuader  : {t['persuader_message']}")
        lines.append(f"Actual     : {t['human_response']}")
        lines.append(f"Predicted  : {t['llm_prediction']}")
        lines.append("")

    return "\n".join(lines)


def extract_json_from_text(text):
    """Extract the first JSON object from model output, handling markdown fences."""
    fenced = re.search(r"```(?:json)?\s*({.*?})\s*```", text, re.DOTALL)
    if fenced:
        return json.loads(fenced.group(1))
    start = text.find("{")
    end   = text.rfind("}")
    if start != -1 and end != -1:
        return json.loads(text[start:end + 1])
    raise ValueError("No JSON object found in judge output.")


def run_judge(prediction_result, model, tokenizer):
    """
    Run the judge model on a single prediction result.
    Returns a parsed JSON annotation dict.
    """
    user_message = build_judge_user_message(prediction_result)

    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {"role": "user",   "content": user_message},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    in_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=JUDGE_MAX_NEW_TOKENS,
            temperature=JUDGE_TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw_output = tokenizer.decode(
        output_ids[0][in_len:],
        skip_special_tokens=True,
    ).strip()

    try:
        annotation = extract_json_from_text(raw_output)
    except (json.JSONDecodeError, ValueError):
        annotation = {
            "conversation_id": prediction_result["conversation_id"],
            "raw_output": raw_output,
            "parse_error": True,
        }

    return annotation

## 13. Run Judge on All Saved Prediction Files

In [ ]:
pred_files = sorted(PRED_DIR.glob("conv_*.json"))
print(f"Found {len(pred_files)} prediction files to judge.\n")

judge_failed = []

for pred_path in tqdm(pred_files, desc="Running judge"):
    with open(pred_path, "r", encoding="utf-8") as f:
        pred_result = json.load(f)

    conv_id = pred_result["conversation_id"]

    try:
        annotation = run_judge(pred_result, judge_model, judge_tokenizer)

        judge_path = JUDGE_DIR / f"judge_conv_{conv_id}.json"
        with open(judge_path, "w", encoding="utf-8") as f:
            json.dump(annotation, f, indent=2, ensure_ascii=False)

        parse_ok = not annotation.get("parse_error", False)
        print(f"  Saved: {judge_path.name}  (parse_ok={parse_ok})")

    except Exception as e:
        print(f"  ERROR judging {conv_id}: {e}")
        judge_failed.append(conv_id)

print(f"\nJudge done. {len(pred_files) - len(judge_failed)}/{len(pred_files)} files annotated.")
if judge_failed:
    print(f"Failed: {judge_failed}")

## 14. Quick Sanity Check — Preview One Result

In [ ]:
preview_pred_path  = sorted(PRED_DIR.glob("conv_*.json"))[0]
conv_id_preview    = preview_pred_path.stem.replace("conv_", "")
preview_judge_path = JUDGE_DIR / f"judge_conv_{conv_id_preview}.json"

with open(preview_pred_path, "r") as f:
    pred_preview = json.load(f)

print(f"=== Conversation {pred_preview['conversation_id']} ===")
print(f"Persuadee: {pred_preview['persuadee_id']}")
print()

for turn in pred_preview["turns"]:
    print(f"[Turn {turn['turn_index']}]")
    print(f"  Persuader : {turn['persuader_message'][:120]}")
    print(f"  Actual    : {turn['human_response'][:120]}")
    print(f"  Predicted : {turn['llm_prediction'][:120]}")
    print()

if preview_judge_path.exists():
    print("=== Judge Annotations ===")
    with open(preview_judge_path, "r") as f:
        judge_preview = json.load(f)
    print(json.dumps(judge_preview, indent=2))
else:
    print("Judge annotation not yet available for this conversation.")